In [ ]:
from datetime import date
import yaml

from src.Functions_Load_Emulator_v2 import *

---

## **Emulate "base load" domestic consumption**
(no induction hob; no pv; no bess; no hvac)

### **1. Initialize load emulator and get calendar**

In this section you need to specify the number of users to emulate and the simulation period.

In [ ]:
#---------------------------------------------------------------------------------------------
# Number of users to emulate
#---------------------------------------------------------------------------------------------

num_user = 100

#---------------------------------------------------------------------------------------------
# Simulation period
#---------------------------------------------------------------------------------------------

start_day = date(2026, 1, 1) # start day for simulation
end_day = date(2026, 12, 31) # end day for simulation

---

### **2. Get calendar**

In this section two different dataframe are created:
- **calendar_df**, a dataframe with the datetime (in format 'YYYY-MM-DD HH:MM:SS'), day_week (from 1-7), day_flag (Monday'', 'Thuesday', etc.) and day_type ('working_day' or 'weekend') information


- **calendar_daily**, a dataframe with the date (in format 'YYYY-MM-DD'), day_week (from 1-7), day_flag (Monday'', 'Thuesday', etc.) and day_type ('working_day' or 'weekend') information

In [ ]:
calendar_df, calendar_daily = create_calendar_dfs(start_day, end_day)

---

### **3. Get input data**

All the input data are imported. The Load Emulator Version 2.0 needs the following data:

- **dict_appliances_load**, a dictionary containing all the load consumption profiles collected for different appliances. To collect these profiles, several open-source and private databases were analyzed (note: for some appliances, a Standard Load Profile (SLP) was collected; in other cases, mean values ​​and standard deviation were collected).

    <span style="color:red">Attention:</span> In this dataframe all the appliance consumption and statistics are named as [*"appliance_name" + "_" + "number"*]. While the endings *"_oc"* and *"_d"* mean  *off cycle duration* ("off cycle" duration statistics) and *duration* ("on cycle" duration statistics) respectively. This statistics are reported only for specific appliance such as fridge, freezer, lamp, etc.

- **dict_appliances_load_info**, a dictionary containing all the useful information to correctly emulate each single appliance.


- **list_appliances_sp**, list of all the "spike profile" appliances, these appliances have a very short usage time and their consumption profile appears to be a "spike" (ex. blender, coffee machine, etc.).
- **list_appliances_blp**, list of all "base load with pattern profile" appliances, they are constantly active and their consumption profile follows specific patterns (ex. fridge, etc.).
- **list_appliances_blnp**, list of all "base load without pattern profile" appliances, they are constantly active and their consumption profile doesn't follow specific patterns (ex. internet router, etc.).
- **list_appliances_dc**, list of all "duty cycle profile" appliances, they have specific standard load profile consumption (ex. washing machine, dishwasher etc.).


- **dict_base_load_stats**, mean and standard deviation profiles for the base load.


- **df_usage_probability_wd**, the quarterly usage probability of activation for each appliance in the 'working_day'.
- **df_usage_probability_we**, the quarterly usage probability of activation for each appliance in the 'weekend'.


- **dict_multi_usage_probability**, the probability to have multiple or zero usages in a single day for a specific appliance.
- **df_multi_usage_probability_wd**, the probability to have multiple or zero usages in a single day for a specific appliance in 'working_day'.
- **df_multi_usage_probability_we**, the probability to have multiple or zero usages in a single day for a specific appliance in 'weekend'.


- **df_clusters_distribution**, different membership clusters were modeled for the users, the clusters differ in number and category of members and this influences the technological equipment of the users. In this dataframe the probability of belonging to each single cluster is reported.
- **df_clusters_equipment**, in this dataframe, the probability of owning a single household appliance is reported for each cluster.
    
    <span style="color:red">Attention:</span> It is very important that for each appliance listed in this dataframe there is a corrispective consumption profile or consumption statistics in the 'dict_appliance_load' (with the same name).
- **df_clusters_equipment_multi**, in this dataframe, the probability of owning multiple device for a single appliance category is reported for each cluster.


- **df_monthly_usage_probability**, in this dataframe, the monthly probability of activation in report for each appliance. Some appliances have typical seasonal consumption (ex. dryer, radiator, etc.).

All the input data can be better explored in the following folder:

📂 *files/energy/input/load_emulator/input_emulator_v2/*

In [ ]:
data_input = import_data_load_emulator_v2()

**Attention:** The following function remove the probability to assign a specific appliance.  

In [ ]:
data_input = remove_specific_appliance(data_input, specific_appliance='induction_hob')

---

### 4. Evaluate daily activation matrix and alculate scheduled consumption for each appliance

The user load consumption is emulated and all results are collected in the *dict_users* dictionary and in the *stacked_df* dataframe.

The information collected in the *dict_users* dictionary can be explored in the 6.1 paragraph of this tutorial.

In the *stacked_df* dataframe the total consumption profiles for each single emulated user are reported.

Parameters explanation:
- **show_results**, if True the plot with all the appliances and the total consumption are shown.


- **save_all_results**, if True the *dict_users* dictionary is exported in the folder *files/energy/input/load_emulator/results_emulator*.

The *stacked_df* is always saved in the folder:

📂 *files/energy/input/load_emulator/results_emulator*.

In [ ]:
dict_users, stacked_df = load_emulator_v2(num_user, 
                              data_input, 
                              calendar_df, 
                              calendar_daily,
                              
                              simulate_boiler=False, # if True, the boiler will be simulated 
                              all_boiler_profiles=True,
                              
                              show_results=False, # a progress bar and some plots with results will be shown
                              save_all_results=False, # save the results in a pickle file, disactivate if there are too many users!!
                              
                              specific_appliance=None, # if not None, only the specified appliance will be simulated (e.g. 'washing_machine', 'induction_hob', 'boiler', etc.)
                              
                              parallelize=True, # parallelize the creation of duty cycle profiles, time-consuming part of the process
                              max_workers=5, # number of workers to use for parallelization
                              )

---

### 5. Calculate and export mean profile

The mean profile is saved in the following folder:

📂 *files/energy/input/load_emulator/results_emulator/mean_profile_load_emulator_v2.csv*.

In [ ]:
# export_mean_profile_load_emulator_v2(stacked_df)

---

### 6. Analyze results

#### 6.1 Explore "dict_users.pkl"

This function is useful to explore all the data collected in the *dict_users.pkl* file.

In [ ]:
# This function is useful to explore the contents of the dictionary
# For an advanced analysis run the script "app_explore_dictionary.py"

# config = yaml.safe_load(open("config.yml", 'r')) 
# path_results_emulator = config['foldername_result_emulator']
# explore_dictionary(path_results_emulator + "dict_users_emulator_v2.pkl")

#### 6.2 analyze results for user

This function is useful to printing some general information of a specific emulated user.

In [ ]:
# user_id = 'user_0'
# analyze_results(user_id, calendar_df, dict_users, data_input['df_clusters_distribution'])

This function is useful for plotting the annual average profile on a quarterly basis for a specific user.

In [ ]:
# df_user_consumption = extract_df_user_consumption(dict_users, user_id, calendar_df)
# plot_average_load_profile(user_id, df_user_consumption, calendar_df)

#### 6.3 Analyze results single appliance

This function is useful for plotting the annual average profile on a quarterly basis for each appliance and for a specific user.

In [ ]:
# plot_average_appliance_load_profile(df_user_consumption, calendar_df)

This function is useful for plotting all the consumption profiles on a quarterly basis for specific appliance and for a specific user.

In [ ]:
# appliance = 'boiler'
# plot_load_profile_by_day_type(df_user_consumption, calendar_df, appliance, hourly_resample = False)

---
---

## **Emulate Induction Hob Profiles**

### 1. Initialize load emulator and get calendar

In this section the consumption profiles for a specific appliance in a specific simulation period are emulated.

*Note:* The *'base_load'* can't be selected for the emulation at the moment.

In [ ]:
#---------------------------------------------------------------------------------------------
# Specific appliace to emulate
#---------------------------------------------------------------------------------------------

specific_appliance = 'induction_hob' # if not None, only the specified appliance will be simulated (e.g. 'washing_machine', 'induction_hob', 'boiler', etc.)

#---------------------------------------------------------------------------------------------
# Number of users to emulate
#---------------------------------------------------------------------------------------------

num_appliance = 1000

#---------------------------------------------------------------------------------------------
# Simulation period
#---------------------------------------------------------------------------------------------

start_day = date(2026, 1, 1) # start day for simulation
end_day = date(2026, 12, 31) # end day for simulation

---

### 2. Get calendar

In [ ]:
calendar_df, calendar_daily = create_calendar_dfs(start_day, end_day)

---

### 3. Get input data

In [ ]:
data_input = import_data_load_emulator_v2()

---

### 4. Emulate only induction hob

In [ ]:
dict_users, stacked_df = load_emulator_v2(num_appliance, 
                              data_input, 
                              calendar_df, 
                              calendar_daily,
                              
                              simulate_boiler=False, # if True, the boiler will be simulated 
                              all_boiler_profiles=True,
                              
                              show_results=False, # a progress bar and some plots with results will be shown
                              save_all_results=True, # save the results in a pickle file, disactivate if there are too many users!!
                              
                              specific_appliance = specific_appliance, # if not None, only the specified appliance will be simulated (e.g. 'washing_machine', 'induction_hob', 'boiler', etc.)
                              
                              parallelize=False, # parallelize the creation of duty cycle profiles, time-consuming part of the process
                              max_workers=1, # number of workers to use for parallelization
                              )
                              

---

### 5. Calculate and export mean profile

The mean profile is saved in the following folder:

📂 *files/energy/input/load_emulator/results_emulator/appliance_name_mean_profile_load_emulator_v2.csv*.

In [ ]:
# export_mean_profile_load_emulator_v2(stacked_df, specific_appliance)

---